In [1]:
import os
os.chdir(r"C:\Users\HP\Desktop\image-search")
print(os.getcwd())

C:\Users\HP\Desktop\image-search


In [2]:
import sys
sys.path.append(".")

In [3]:
import aiohttp
import numpy as np
from tqdm import tqdm
from src.utils import load_image_from_url
from models.embedder import embed_images_batch
import asyncio

c:\Users\HP\Desktop\image-search\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
import pandas as pd

df = pd.read_csv(
    "data/unsplash_lite/photos.csv",
    sep="\t",
    on_bad_lines="skip"
)

print(len(df))
print(df.columns)

25000
Index(['photo_id', 'photo_url', 'photo_image_url', 'photo_submitted_at',
       'photo_featured', 'photo_width', 'photo_height', 'photo_aspect_ratio',
       'photo_description', 'photographer_username', 'photographer_first_name',
       'photographer_last_name', 'exif_camera_make', 'exif_camera_model',
       'exif_iso', 'exif_aperture_value', 'exif_focal_length',
       'exif_exposure_time', 'photo_location_name', 'photo_location_latitude',
       'photo_location_longitude', 'photo_location_country',
       'photo_location_city', 'stats_views', 'stats_downloads',
       'ai_description', 'ai_primary_landmark_name',
       'ai_primary_landmark_latitude', 'ai_primary_landmark_longitude',
       'ai_primary_landmark_confidence', 'blur_hash'],
      dtype='object')


In [5]:
df = df.dropna(subset=["photo_image_url"])
df = df[df["photo_image_url"].str.startswith("https")]
df = df.reset_index(drop=True)

print("Total usable:", len(df))

Total usable: 25000


In [6]:
LIMIT = 10000
df = df.iloc[:LIMIT]

In [7]:
async def run_batched(df, limit=10000, batch_size=32):

    embeddings = []
    ids = []

    connector = aiohttp.TCPConnector(limit=20)
    timeout = aiohttp.ClientTimeout(total=10)

    df = df.iloc[:limit]

    batch_images = []
    batch_ids = []

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:

        for _, row in tqdm(df.iterrows(), total=len(df)):

            try:
                img = await load_image_from_url(session, row["photo_image_url"])
            except:
                continue

            if img is None:
                continue

            batch_images.append(img)
            batch_ids.append(row["photo_id"])

            if len(batch_images) >= batch_size:
                vecs = embed_images_batch(batch_images)

                embeddings.append(vecs)
                ids.extend(batch_ids)

                batch_images.clear()
                batch_ids.clear()

        # flush remaining
        if batch_images:
            vecs = embed_images_batch(batch_images)
            embeddings.append(vecs)
            ids.extend(batch_ids)

    embeddings = np.concatenate(embeddings, axis=0).astype("float32")

    return ids, embeddings

In [8]:
ids, embeddings = await run_batched(df, batch_size=32)

  7%|▋         | 665/10000 [11:55<3:59:37,  1.54s/it] c:\Users\HP\Desktop\image-search\.venv\Lib\site-packages\PIL\Image.py:3451: DecompressionBombWarning: Image size (96012000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
 13%|█▎        | 1295/10000 [24:55<1:16:59,  1.88it/s] c:\Users\HP\Desktop\image-search\.venv\Lib\site-packages\PIL\Image.py:3451: DecompressionBombWarning: Image size (99996120 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
 24%|██▍       | 2413/10000 [47:11<2:19:49,  1.11s/it] c:\Users\HP\Desktop\image-search\.venv\Lib\site-packages\PIL\Image.py:3451: DecompressionBombWarning: Image size (107184040 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
 34%|███▍      | 3441/10000 [1:07:09<47:07,  2.32it/s]   c:\Users\HP\Desktop\image-search\.venv\Lib\site-packages\PIL\Image.py:3451: DecompressionBombWarning: Image size

In [1]:
print(embeddings.shape, len(ids))

# NaN check
print("NaNs:", np.isnan(embeddings).any())

# Norms
norms = np.linalg.norm(embeddings, axis=1)
print(norms[:10])
print("Min norm:", norms.min(), "Max norm:", norms.max())

NameError: name 'embeddings' is not defined

In [21]:
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

In [ ]:
import faiss

d = embeddings.shape[1]

index = faiss.IndexFlatIP(d)  # cosine similarity (after normalization)
index.add(embeddings)

print("Index size:", index.ntotal)

Index size: 9642


In [23]:
import random

i = random.randint(0, len(ids)-1)

query_vec = embeddings[i].reshape(1, -1)

distances, indices = index.search(query_vec, k=5)

print(indices)
print(distances)

[[6477 5821 7704 4399 1131]]
[[1.0000001  0.94299626 0.93491405 0.93483925 0.93006027]]


In [24]:
results = [ids[idx] for idx in indices[0]]
print("Query ID:", ids[i])
print("Results:", results)

Query ID: kwZKWu3OqzU
Results: ['kwZKWu3OqzU', 'rSjgPKsBSkY', 'CztHr_SZq7E', 'hCHws7MnJFQ', 'XxXAkVuEnvU']


In [25]:
np.save("data/unsplash/embeddings.npy", embeddings)
np.save("data/unsplash/ids.npy", np.array(ids))
faiss.write_index(index, "data/unsplash/index.faiss")

In [26]:
def search_by_index(i, k=5):
    query = embeddings[i].reshape(1, -1)
    distances, indices = index.search(query, k)

    print("Query ID:", ids[i])
    print("Results:", [ids[idx] for idx in indices[0]])
    print("Scores:", distances[0])

In [27]:
search_by_index(0)
search_by_index(50)
search_by_index(200)

Query ID: oSf8ePoG9NU
Results: ['oSf8ePoG9NU', 'Sza0bpjd8gw', 'd5SALKdoGb4', 'M6a5BrXz1gU', 'VhnFatTrDIU']
Scores: [1.0000002  0.90054643 0.8998729  0.87782735 0.87560064]
Query ID: upT9R6CO-Hc
Results: ['upT9R6CO-Hc', 'JE5g2ZZG9qA', 'hEDwwsfES2w', 'tUDlGlr1sAA', 'kDo57E2qOE4']
Scores: [1.         0.9420435  0.93422604 0.9299308  0.92592883]
Query ID: CKJYAcSc3WE
Results: ['CKJYAcSc3WE', 'gQrYB3j9mJ0', 'XF-ZLS5G0QU', 'Y1ByvAGQ5iE', '8wV8bdQtmTo']
Scores: [0.99999994 0.9747372  0.9696759  0.9604498  0.9560792 ]


In [28]:
search_by_index(100)
search_by_index(101)
search_by_index(102)

Query ID: SoC1ex6sI4w
Results: ['SoC1ex6sI4w', 'MXJ3QUxhNrY', 'm78EF2Rnp9A', '15mkCCDr9iY', 'Rj2Xuc6-cd0']
Scores: [0.9999999  0.8927495  0.8846672  0.88326585 0.87699264]
Query ID: pPxdiv5MkS0
Results: ['pPxdiv5MkS0', '1PCoQuJspGw', 'q1X6ADPZl4w', '0W_UkUJY2hA', '9CfajiGQL0o']
Scores: [1.0000001  0.9221769  0.92163104 0.92136544 0.91924536]
Query ID: _ppnPXy_TVw
Results: ['_ppnPXy_TVw', '_iMJcYw6bJ0', '1ssk3G5f1Kw', 'sirEpWjfSmo', 'jx_kpR7cvDc']
Scores: [1.         0.92164314 0.9194994  0.9119355  0.9111342 ]


In [29]:
import random

for _ in range(5):
    search_by_index(random.randint(0, 999))

Query ID: Syl05eRC0nA
Results: ['Syl05eRC0nA', 'KHTN7BtUPhM', 'dLgh-IdELuM', 'g7jLUB8WlWk', 'S4G-9Ua-sfM']
Scores: [1.0000002 0.9450349 0.9395633 0.9386977 0.9384346]
Query ID: rRGTgjHQHDQ
Results: ['rRGTgjHQHDQ', 'eWeteZ9W3e0', 'CBa301Yn7F8', 'wAJJJbP7mbc', '2YbEDUdNbDM']
Scores: [1.        0.9309693 0.919662  0.9089951 0.9071591]
Query ID: VouUERkgtzM
Results: ['VouUERkgtzM', 'I-Rx35qlxPU', '8SivQyBVe2A', 'fdR2GOe4aBY', 'xuqp36-VuDM']
Scores: [1.        0.8521692 0.8340891 0.8151623 0.8036047]
Query ID: NSSsyfAQW2g
Results: ['NSSsyfAQW2g', 'zufAJSV_8zA', '8V-5Y_Mdbn4', 'g-KH52m2P7g', 'NXH0kizvhB4']
Scores: [1.0000001  0.9575504  0.8755354  0.83367985 0.8336227 ]
Query ID: P3PUwBUNnMA
Results: ['P3PUwBUNnMA', 'nD90C0qxVbs', 'PSUOqQQqyrw', 'tmsxaFx1Sws', 'VwUHNE7KZKg']
Scores: [1.         0.91052663 0.90378    0.89000773 0.881899  ]
